# 1. Data Generation
## E-Commerce Order Analytics System


In [1]:
import random
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
RAW_DIR = BASE /"data"/"raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

N_CUSTOMERS = 800
N_PRODUCTS = 500
N_ORDERS = 2000
N_ORDER_ITEMS = 5000

DATA_START = datetime(2024, 7, 1)
DATA_END = datetime(2026, 7, 10)
print(f"Generating the data that covers date from {DATA_START:%Y-%m-%d} to {DATA_END:%Y-%m-%d}")

Generating the data that covers date from 2024-07-01 to 2026-07-10


## 2. customers.csv

The customer type is not the same, for everyone most of the customers are customers. The dates when people registered go from January 2024. Continue from there. This means the query that analyzes groups of customers has to deal with groups. Some email addresses are not correct 2 percent of them and these are added at the end: half of the bad email addresses are missing the @ symbol and the other half are missing the domain part of the email address.


In [2]:
CUSTOMER_TYPES = ["REGULAR", "PREMIUM", "VIP"]
CUSTOMER_TYPE_WEIGHTS = [0.70, 0.20, 0.10]

customers = []
for i in range(1, N_CUSTOMERS + 1):
    name = fake.name()
    email_user = name.lower().replace(" ", ".").replace("'", "")
    email = f"{email_user}@{fake.free_email_domain()}"
    reg_date = fake.date_between_dates(
        date_start=datetime(2024, 1, 1), date_end=datetime(2026, 5, 31)
    )
    customers.append({
        "customer_id": f"C{i:04d}",
        "customer_name": name,
        "email": email,
        "registration_date": reg_date.strftime("%Y-%m-%d"),
        "customer_type": random.choices(CUSTOMER_TYPES, CUSTOMER_TYPE_WEIGHTS)[0],
    })

customers_df = pd.DataFrame(customers)

n_bad_emails = int(N_CUSTOMERS * 0.02)
bad_email_idx = random.sample(range(N_CUSTOMERS), n_bad_emails)
for j, idx in enumerate(bad_email_idx):
    email = customers_df.loc[idx,"email"]
    if j % 2 == 0:
        customers_df.loc[idx, "email"] = email.replace("@", "")
    else:
        customers_df.loc[idx, "email"] = email.split("@")[0] + "@"

print(f"customers_count:{len(customers_df)} rows,{n_bad_emails} invalid emails are injected")
customers_df.head()

customers_count:800 rows,16 invalid emails are injected


,customer_id,customer_name,email,registration_date,customer_type
0,C0001,Allison Hill,allison.hill@gmail.com,2024-08-04,REGULAR
1,C0002,Megan Mcclain,megan.mcclain@gmail.com,2025-10-17,REGULAR
2,C0003,Brandon Hall,brandon.hall@hotmail.com,2024-02-11,REGULAR
3,C0004,Michelle Miles,michelle.miles@yahoo.com,2024-12-08,REGULAR
4,C0005,Donald Booth,donald.booth@gmail.com,2025-10-18,PREMIUM


## 3. products.csv

Product names are made up of words that're specific to a certain category. Six percent of these names have extra spaces or the letters are not capitalized correctly. This problem will be fixed later by the function called clean_products().

In [3]:
CATALOG = {
    "Electronics": {
        "subcategories": ["Mobiles", "Laptops", "Audio", "Accessories", "Cameras"],
        "nouns": ["Smartphone", "Laptop", "Headphones", "Speaker", "Charger",
                  "Keyboard", "Mouse", "Monitor", "Camera", "Tablet", "Smartwatch",
                  "Power Bank", "Earbuds", "Webcam", "Router"],
    },
    "Clothing": {
        "subcategories": ["Men", "Women", "Kids", "Footwear", "Sportswear"],
        "nouns": ["T-Shirt", "Jeans", "Jacket", "Sneakers", "Hoodie", "Dress",
                  "Sweater", "Shorts", "Cap", "Socks", "Scarf", "Blazer"],
    },
    "Home": {
        "subcategories": ["Kitchen", "Furniture", "Decor", "Bedding", "Storage"],
        "nouns": ["Blender", "Cookware Set", "Lamp", "Cushion", "Bookshelf",
                  "Curtains", "Bedsheet", "Organizer", "Vase", "Wall Clock",
                  "Coffee Maker", "Dinner Set"],
    },
    "Books": {
        "subcategories": ["Fiction", "Non-Fiction", "Academic", "Comics", "Self-Help"],
        "nouns": ["Novel", "Biography", "Textbook", "Comic Book", "Cookbook",
                  "Guidebook", "Anthology", "Encyclopedia", "Journal", "Atlas"],
    },
}
ADJECTIVES = ["Classic", "Premium", "Eco", "Smart", "Ultra", "Compact", "Deluxe",
              "Pro", "Essential", "Modern", "Vintage", "Portable"]

products = []
for i in range(1, N_PRODUCTS + 1):
    category = random.choice(list(CATALOG.keys()))
    sub = random.choice(CATALOG[category]["subcategories"])
    noun = random.choice(CATALOG[category]["nouns"])
    brand = fake.last_name()
    name = f"{brand} {random.choice(ADJECTIVES)} {noun}"
    cost = round(random.uniform(3, 800), 2)
    products.append({
        "product_id": f"P{i:04d}",
        "product_name": name,
        "category": category,
        "subcategory": sub,
        "cost_price": cost,
    })

products_df = pd.DataFrame(products)

n_messy = int(N_PRODUCTS * 0.06)
messy_idx = random.sample(range(N_PRODUCTS), n_messy)
for j, idx in enumerate(messy_idx):
    name = products_df.loc[idx, "product_name"]
    style = j % 3
    if style == 0:
        products_df.loc[idx, "product_name"] = f"  {name.lower()}  "
    elif style == 1:
        products_df.loc[idx, "product_name"] = name.upper()
    else:
        products_df.loc[idx, "product_name"] = name.replace(" ", "  ")

print(f"products_count:{len(products_df)}rows,{n_messy} messy names are injected")
products_df.head()

products_count:500rows,30 messy names are injected


,product_id,product_name,category,subcategory,cost_price
0,P0001,Ayala Smart Guidebook,Books,Comics,784.68
1,P0002,Garrett Smart Sweater,Clothing,Sportswear,685.24
2,P0003,Cameron Portable Guidebook,Books,Fiction,379.96
3,P0004,Turner Pro Textbook,Books,Comics,771.97
4,P0005,Tyler Compact Socks,Clothing,Sportswear,696.15


## 4. orders.csv

Every order has a customer_id` which makes sense because its like that by design.

Then we add some issues:
- About **5%** of `customer_id` values are either NULL or empty (we do both as per the requirements).

- Roughly **3%** of dates are in the format, which is `DD-MM-YYYY`.

- There are **3 orders** with a order_date` just to test some unusual cases.

Also orders can only be placed **on or after** the customers registration date. This way our cohort analysis looks more realistic.
We use customer_id` values for every order.
The orders are placed on or, after the customers registration date.
This helps our analysis work properly.

In [4]:
STATUSES = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
STATUS_WEIGHTS = [0.15, 0.20, 0.50, 0.08, 0.07]
REGIONS = ["NORTH", "SOUTH", "EAST", "WEST", "CENTRAL"]

reg_lookup = dict(zip(customers_df["customer_id"],
                      pd.to_datetime(customers_df["registration_date"])))

orders = []
for i in range(1, N_ORDERS + 1):
    cust_id = random.choice(customers_df["customer_id"].tolist())
    # order must come after the customer registered
    start = max(DATA_START, reg_lookup[cust_id].to_pydatetime())
    span_seconds = max(int((DATA_END - start).total_seconds()), 3600)
    order_dt = start + timedelta(seconds=random.randint(0, span_seconds))
    orders.append({
        "order_id": f"O{i:05d}",
        "customer_id": cust_id,
        "order_date": order_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "status": random.choices(STATUSES, STATUS_WEIGHTS)[0],
        "region_code": random.choice(REGIONS),
    })

orders_df = pd.DataFrame(orders)

n_null_cust = int(N_ORDERS * 0.05)
null_idx = random.sample(range(N_ORDERS), n_null_cust)
for j, idx in enumerate(null_idx):
    orders_df.loc[idx, "customer_id"] = "NULL" if j % 2 == 0 else ""

n_bad_dates = int(N_ORDERS * 0.03)
remaining = [i for i in range(N_ORDERS) if i not in null_idx]
bad_date_idx = random.sample(remaining, n_bad_dates)
for idx in bad_date_idx:
    dt = datetime.strptime(orders_df.loc[idx, "order_date"], "%Y-%m-%d %H:%M:%S")
    orders_df.loc[idx, "order_date"] = dt.strftime("%d-%m-%Y %H:%M:%S")

future_idx = random.sample([i for i in remaining if i not in bad_date_idx], 3)
for idx in future_idx:
    future_dt = DATA_END + timedelta(days=random.randint(200, 400))
    orders_df.loc[idx, "order_date"] = future_dt.strftime("%Y-%m-%d %H:%M:%S")

print(f"orders_count:{len(orders_df)} rows, {n_null_cust} NULL/empty customer_id, "
      f"{n_bad_dates} wrong-format dates, 3 future dates are injected")
orders_df.head()

orders_count:2000 rows, 100 NULL/empty customer_id, 60 wrong-format dates, 3 future dates are injected


,order_id,customer_id,order_date,status,region_code
0,O00001,C0353,2026-04-29 16:15:01,DELIVERED,WEST
1,O00002,C0277,2025-10-20 16:00:39,PLACED,CENTRAL
2,O00003,C0225,2026-03-13 16:33:50,CANCELLED,EAST
3,O00004,C0060,2026-02-15 08:50:03,DELIVERED,CENTRAL
4,O00005,C0239,2025-08-31 23:37:19,PLACED,SOUTH


## 5. order_items.csv

Each item is given an order id and product id. The unit price of each item is figured out from the products cost price by adding a markup of 20 to 80 percent. There are some problems with the items:
- Some items have a quantity, which is like a return and this happens about 3 percent of the time.
- There are 15 items that have order ids that do not exist like O99xxx, which's not good because it breaks the rules of how the orders and items should be connected.
- There are 5 rows where the discount percent's more than 100 and there are also 5 rows where the quantity of the item is 0, which are some really unusual cases, for the items and orders.

In [5]:
cost_lookup = dict(zip(products_df["product_id"], products_df["cost_price"]))
order_ids = orders_df["order_id"].tolist()
product_ids = products_df["product_id"].tolist()
DISCOUNTS = [0, 0, 0, 5, 10, 10, 15, 20, 25, 30]  # weighted toward small discounts

items = []
for i in range(1, N_ORDER_ITEMS + 1):
    pid = random.choice(product_ids)
    unit_price = round(cost_lookup[pid] * random.uniform(1.2, 1.8), 2)
    items.append({
        "item_id": f"I{i:05d}",
        "order_id": random.choice(order_ids),
        "product_id": pid,
        "quantity": random.randint(1, 5),
        "unit_price": unit_price,
        "discount_percent": random.choice(DISCOUNTS),
    })

items_df = pd.DataFrame(items)

n_returns = int(N_ORDER_ITEMS * 0.03)
return_idx = random.sample(range(N_ORDER_ITEMS), n_returns)
items_df.loc[return_idx, "quantity"] = -items_df.loc[return_idx, "quantity"]


problem_products = random.sample(product_ids, 3)
heavy_return_rows = random.sample(return_idx, 36)
for j, idx in enumerate(heavy_return_rows):
    pid = problem_products[j % 3]
    items_df.loc[idx, "product_id"] = pid
    items_df.loc[idx, "quantity"] = -random.randint(4, 6)
    items_df.loc[idx, "unit_price"] = round(cost_lookup[pid] * random.uniform(1.2, 1.8), 2)
print(f"problem products (heavy returns): {problem_products}")

pool = [i for i in range(N_ORDER_ITEMS) if i not in return_idx]

orphan_idx = random.sample(pool, 15)
for j, idx in enumerate(orphan_idx):
    items_df.loc[idx, "order_id"] = f"O99{900 + j}"
pool = [i for i in pool if i not in orphan_idx]

bad_disc_idx = random.sample(pool, 5)
items_df.loc[bad_disc_idx, "discount_percent"] = [110, 150, 125, 175, 200]
pool = [i for i in pool if i not in bad_disc_idx]

zero_qty_idx = random.sample(pool, 5)
items_df.loc[zero_qty_idx, "quantity"] = 0

print(f"order_item_count: {len(items_df)} rows, {n_returns} returns (negative qunatity), "
      f"15 orphan order_ids, 5 discount>100, 5 zero-quantity are injected")
items_df.head()

problem products (heavy returns): ['P0439', 'P0462', 'P0481']
order_item_count: 5000 rows, 150 returns (negative qunatity), 15 orphan order_ids, 5 discount>100, 5 zero-quantity are injected


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,I00001,O01278,P0255,5,608.43,0
1,I00002,O01191,P0325,1,957.20,10
2,I00003,O00457,P0437,4,875.35,25
3,I00004,O01754,P0196,4,625.09,0
4,I00005,O01872,P0180,4,1305.92,15


## 6. Saving all four CSVs files

In [6]:
customers_df.to_csv(RAW_DIR / "customers.csv", index=False)
products_df.to_csv(RAW_DIR / "products.csv", index=False)
orders_df.to_csv(RAW_DIR / "orders.csv", index=False)
items_df.to_csv(RAW_DIR / "order_items.csv", index=False)

for f in sorted(RAW_DIR.glob("*.csv")):
    print(f"{f.name:20s} {f.stat().st_size / 1024:8.1f} KB")

customers.csv            50.6 KB
order_items.csv         159.4 KB
orders.csv               95.6 KB
products.csv             25.8 KB


## 7. Verification - this confirms every required issue is present at the right rate

In [7]:
raw_orders = pd.read_csv(RAW_DIR / "orders.csv", dtype=str, keep_default_na=False)
raw_items = pd.read_csv(RAW_DIR / "order_items.csv")
raw_customers = pd.read_csv(RAW_DIR / "customers.csv")
raw_products = pd.read_csv(RAW_DIR / "products.csv")

null_cust = ((raw_orders["customer_id"] == "NULL") | (raw_orders["customer_id"] == "")).sum()
wrong_fmt = raw_orders["order_date"].str.match(r"\d{2}-\d{2}-\d{4}").sum()
neg_qty = (raw_items["quantity"] < 0).sum()
orphans = (~raw_items["order_id"].isin(raw_orders["order_id"])).sum()
bad_email = (~raw_customers["email"].str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")).sum()
messy_names = (raw_products["product_name"] != raw_products["product_name"].str.strip().str.title()).sum()

checks = pd.DataFrame([
    ("NULL/empty customer_id in orders", null_cust, f"{null_cust / len(raw_orders):.1%}", "target 5%"),
    ("Wrong date format (DD-MM-YYYY)", wrong_fmt, f"{wrong_fmt / len(raw_orders):.1%}", "target ~3%"),
    ("Negative quantity (returns)", neg_qty, f"{neg_qty / len(raw_items):.1%}", "target 3%"),
    ("Orphan order_ids in order_items", orphans, "-", "target 15"),
    ("Invalid emails", bad_email, f"{bad_email / len(raw_customers):.1%}", "target 2%"),
    ("Messy product names", messy_names, f"{messy_names / len(raw_products):.1%}", "target ~6%"),
    ("discount_percent > 100", (raw_items["discount_percent"] > 100).sum(), "-", "target 5"),
    ("quantity = 0", (raw_items["quantity"] == 0).sum(), "-", "target 5"),
], columns=["issue", "count", "rate", "expected"])

assert len(raw_orders) >= 500 and len(raw_items) >= 500
assert len(raw_customers) >= 500 and len(raw_products) >= 500
print("All 4 files have >= 500 rows")
checks

All 4 files have >= 500 rows


,issue,count,rate,expected
0,NULL/empty customer_id in orders,100,5.0%,target 5%
1,Wrong date format (DD-MM-YYYY),60,3.0%,target ~3%
2,Negative quantity (returns),150,3.0%,target 3%
3,Orphan order_ids in order_items,15,-,target 15
4,Invalid emails,16,2.0%,target 2%
5,Messy product names,20,4.0%,target ~6%
6,discount_percent > 100,5,-,target 5
7,quantity = 0,5,-,target 5
